In [1]:
#setup
import pandas as pd
import numpy as np
import glob, os
from IPython.display import display
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
file_path = "/content/drive/MyDrive/mapping_mortality_gap/data_clean/national_microdata_clean.csv"
df = pd.read_csv(file_path)

In [3]:
#groupping the rows to find death count by year, age, sex, and race

death_counts = (
    df.groupby(["year", "age_group", "sex", "race_category", "cause_of_death"]).size().reset_index(name="deaths")
)
death_counts["year"] = death_counts["year"].astype(int)
death_counts.head(10)

,year,age_group,sex,race_category,cause_of_death,deaths
0,2018,0-24,Female,Black,Cardiomyopathy (I42),21
1,2018,0-24,Female,Japanese,Cardiomyopathy (I42),1
2,2018,0-24,Female,Others,Cardiomyopathy (I42),3
3,2018,0-24,Female,White,Cardiomyopathy (I42),46
4,2018,0-24,Male,American Indian and Alaskan Native,Cardiomyopathy (I42),2
5,2018,0-24,Male,Asian Indian,Cardiomyopathy (I42),2
6,2018,0-24,Male,Black,Cardiomyopathy (I42),62
7,2018,0-24,Male,Japanese,Cardiomyopathy (I42),3
8,2018,0-24,Male,Others,Cardiomyopathy (I42),6
9,2018,0-24,Male,White,Cardiomyopathy (I42),108


In [4]:
#combining the population by age and sex from all years (2018 to 2024) census data

folder_path = "/content/drive/MyDrive/mapping_mortality_gap/raw_data/census data/2018 - 2024 population by age and sex"
csv_files = glob.glob(os.path.join(folder_path, "*.csv")) #finding all csv files from folder and storing as list
all_dataframes = []

for csv in csv_files:
  temp_df = pd.read_csv(csv, skiprows=[1])
  file_name = os.path.basename(csv)
  year_digits = "".join(filter(str.isdigit, file_name))[:4] #selecting first 4 digits (year) from file name
  temp_df["year"] = int(year_digits)
  all_dataframes.append(temp_df)

#combining all DataFrame into single DataFrame
combined_year_age = pd.concat(all_dataframes, ignore_index=True)


#cleaning inivisble spaces from Label (Grouping) column
combined_year_age["Label (Grouping)"] = combined_year_age["Label (Grouping)"].str.strip()


#droppig the unwanted columns
columns_to_drop = [
    "United States!!Margin of Error", "United States!!Percent", "United States!!Percent Margin of Error"
]

pop_age_sex = combined_year_age.drop(columns= columns_to_drop)

#renaming the columns
pop_age_sex = pop_age_sex.rename(columns={
    "Label (Grouping)" : "category",
    "United States!!Estimate": "population"
})


#removing comma from population and converting into number
pop_age_sex["population"] = pop_age_sex["population"].astype(str).str.replace(",", "")
pop_age_sex["population"] = pd.to_numeric(pop_age_sex["population"], errors = "coerce")


# matching the census data with our cleaned CDC data
category_mapping = {
    'Under 5 years': '0-24',
    '5 to 9 years': '0-24',
    '10 to 14 years': '0-24',
    '15 to 19 years': '0-24',
    '20 to 24 years': '0-24',
    '25 to 34 years': '25-44',
    '35 to 44 years': '25-44',
    '45 to 54 years': '45-64',
    '55 to 59 years': '45-64',
    '60 to 64 years': '45-64',
    '65 to 74 years': '65+',
    '75 to 84 years': '65+',
    '85 years and over': '65+',

    #race (we will be using broad, detailed race for those Asian and Pacific Islander will be done seperatly)
    "White": "White",
    "Black or African American": "Black",
    "American Indian and Alaska Native": "American Indian and Alaskan Native",
    "Asian": "Asian",
    'Native Hawaiian and Other Pacific Islander': 'Native Hawaiian and Other Pacific Islander',
    "Two or More Races": "Others"
}

pop_age_sex["standardized_category"] = pop_age_sex["category"].map(category_mapping)

pop_age_sex = pop_age_sex.dropna(subset=["standardized_category"]).copy()

clean_age_sex = pop_age_sex.groupby(["year", "standardized_category"]).sum()["population"].reset_index()
clean_age_sex.head() # Racial groups must be divided by 2, only the age groups population is correct here (CDC listing broad category as twice.)

,year,standardized_category,population
0,2018,0-24,104008592.0
1,2018,25-44,86843127.0
2,2018,45-64,83892606.0
3,2018,65+,52423114.0
4,2018,American Indian and Alaskan Native,8511997.0


In [5]:
#combining all asian

folder_path = "/content/drive/MyDrive/mapping_mortality_gap/raw_data/census data/2018 - 2024 Detailed Asian population"
asian_csv = glob.glob(os.path.join(folder_path, "*.csv"))
asian_allList = []

for csv in asian_csv:
  temp_df = pd.read_csv(csv, skiprows=[1])
  file_name = os.path.basename(csv)
  years = "".join(filter(str.isdigit, file_name))[:4]
  temp_df["year"] = int(years)
  asian_allList.append(temp_df)

asian_combined = pd.concat(asian_allList, ignore_index=True)

#remove trailing space from label column
asian_combined["Label (Grouping)"] = asian_combined["Label (Grouping)"].str.strip()

#dropping the column
asian_combined = asian_combined.drop(columns=["United States!!Margin of Error"])

#rename the columns
asian_combined = asian_combined.rename(columns={
    "Label (Grouping)": "category",
    "United States!!Estimate": "population"
})

#remove comma from population and change to number
asian_combined["population"] = asian_combined["population"].astype(str).str.replace(",", "")
asian_combined["population"] = pd.to_numeric(asian_combined["population"], errors="coerce")

#matching asian detailed with CDC category
asian_mapping = {
    'Chinese, except Taiwanese': 'Chinese',
    'Hmong': 'Other Asian',
    'Japanese': 'Japanese',
    'Korean': 'Korean',
    'Mongolian': 'Other Asian',
    'Okinawan': 'Japanese',
    'Taiwanese': 'Chinese',
    'Burmese': 'Other Asian',
    'Cambodian': 'Vietnamese',
    'Filipino': 'Filipino',
    'Indonesian': 'Filipino',
    'Laotian': 'Other Asian',
    'Malaysian': 'Filipino',
    'Mien': 'Other Asian',
    'Singaporean': 'Chinese',
    'Thai': 'Other Asian',
    'Vietnamese': 'Vietnamese',
    'Bangladeshi': 'Asian Indian',
    'Bhutanese': 'Asian Indian',
    'Nepalese': 'Asian Indian',
    'Pakistani': 'Asian Indian',
    'Sri Lankan': 'Asian Indian',
    'Afghan': 'Other Asian',
    'Kazakh': 'Other Asian',
    'Uzbek': 'Other Asian',
}

asian_combined["standardized_category"] = asian_combined["category"].map(asian_mapping)

#remove remaining rows that has no need
asian_combined = asian_combined.dropna(subset=["standardized_category"]).copy()

clean_asian = asian_combined.groupby(["year", "standardized_category"]).sum()["population"].reset_index()

clean_asian.head(10)


,year,standardized_category,population
0,2018,Asian Indian,941595.0
1,2018,Chinese,4395912.0
2,2018,Filipino,3011610.0
3,2018,Japanese,786491.0
4,2018,Korean,1468279.0
5,2018,Other Asian,927912.0
6,2018,Vietnamese,2102774.0
7,2019,Asian Indian,965173.0
8,2019,Chinese,4404678.0
9,2019,Filipino,3086314.0


In [6]:
#combining native islanders

folder_path = "/content/drive/MyDrive/mapping_mortality_gap/raw_data/census data/2018 - 2024 detailed Native Hawaiian and Other Pacific Islander popln"
island_csv = glob.glob(os.path.join(folder_path, "*.csv"))
island_allList = []

for island in island_csv:
  temp_df = pd.read_csv(island, skiprows=[1])
  file_name = os.path.basename(island)
  year = "".join(filter(str.isdigit, file_name))[:4]
  temp_df["year"] = int(year)
  island_allList.append(temp_df)

island_combined = pd.concat(island_allList, ignore_index=True)

#remove the trailing spaces
island_combined["Label (Grouping)"] = island_combined["Label (Grouping)"].str.strip()

#dropping the columns
island_combined = island_combined.drop(columns=["United States!!Margin of Error"])

#rename the columns
island_combined = island_combined.rename(columns={
    "Label (Grouping)": "category",
    "United States!!Estimate": "population"
})

#removing comma from population and changing to number
island_combined["population"] = island_combined["population"].astype(str).str.replace(",", "")
island_combined["population"] = pd.to_numeric(island_combined["population"], errors = "coerce")

#matching the island category with CDC
island_mapping = {
    'Native Hawaiian': 'Hawaiian',
    'Samoan': 'Samoan',
    'Tongan': 'Samoan',
    'Chamorro': 'Guamanian or Chamorro',
    'Guamanian': 'Guamanian or Chamorro',
    'Marshallese': 'Other Pacific Islander',
    'Fijian': 'Other Pacific Islander',
    'Chuukese': 'Other Pacific Islander',
}

island_combined["standardized_category"] = island_combined["category"].map(island_mapping)

#remove all unnecessary row values
island_combined = island_combined.dropna(subset=["standardized_category"]).copy()

clean_island = island_combined.groupby(["year", "standardized_category"]).sum()["population"].reset_index()
clean_island.head(10)


,year,standardized_category,population
0,2018,Hawaiian,186996.0
1,2018,Other Pacific Islander,67815.0
2,2018,Samoan,151144.0
3,2019,Hawaiian,198734.0
4,2019,Other Pacific Islander,68441.0
5,2019,Samoan,155935.0
6,2020,Guamanian or Chamorro,81898.0
7,2020,Hawaiian,185474.0
8,2020,Other Pacific Islander,70084.0
9,2020,Samoan,152132.0


In [7]:
#combining all of the census data into single DataFrame
census_combined = pd.concat([clean_asian, clean_island], ignore_index=True)

detailed_race = census_combined["standardized_category"].unique()

# Targetting only young population for race
target_age = ["0-24","25-44"]
deathCount_above = death_counts[death_counts["age_group"].isin(target_age)].copy()
detailed_race = deathCount_above[deathCount_above["race_category"].isin(detailed_race)].copy()

#adding the death count for each race per year
asian_island_death = detailed_race.groupby(["year", "race_category"]).sum()["deaths"].reset_index()

asian_island_death.head()


,year,race_category,deaths
0,2018,Asian Indian,4
1,2018,Filipino,2
2,2018,Japanese,15
3,2019,Asian Indian,4
4,2019,Chinese,1


In [8]:
#the grand merdger for asian and islanders
final_asian_island = pd.merge(
    asian_island_death,
    census_combined,
    left_on=['year', "race_category"],
    right_on=["year", "standardized_category"],
    how="inner"
)

final_asian_island.head()



,year,race_category,deaths,standardized_category,population
0,2018,Asian Indian,4,Asian Indian,941595.0
1,2018,Filipino,2,Filipino,3011610.0
2,2018,Japanese,15,Japanese,786491.0
3,2019,Asian Indian,4,Asian Indian,965173.0
4,2019,Chinese,1,Chinese,4404678.0


In [9]:
#calculating the mortality rate for Asian & islander per 100k

final_asian_island["mortality_per_100k"] = (final_asian_island['deaths']/final_asian_island['population'])*100000
final_asian_island['mortality_per_100k'] = final_asian_island['mortality_per_100k'].round(1)

#removing the redundent column
final_asian_island = final_asian_island.drop(columns=["standardized_category"])
final_asian_island.head()


,year,race_category,deaths,population,mortality_per_100k
0,2018,Asian Indian,4,941595.0,0.4
1,2018,Filipino,2,3011610.0,0.1
2,2018,Japanese,15,786491.0,1.9
3,2019,Asian Indian,4,965173.0,0.4
4,2019,Chinese,1,4404678.0,0.0


In [10]:
#combining with other races like White, Black, American indians
broad_races = ['White', 'Black', 'American Indian and Alaskan Native', 'Others']
broad_populn = clean_age_sex[clean_age_sex["standardized_category"].isin(broad_races)].copy()

#We divide by 2 to cut the bloat and get the true baseline population
broad_populn["population"] = broad_populn["population"]/2


#Filtering deaths for young population only
target_age = ["0-24","25-44"]
broad_death = death_counts[
    (death_counts["race_category"].isin(broad_races)) &
    (death_counts["age_group"].isin(target_age))
].copy()


broad_death_grouped = broad_death.groupby(["year", "race_category"]).sum()["deaths"].reset_index()


#merdging death with population together in single table
broad_merdge = pd.merge(
    broad_death_grouped,
    broad_populn,
    left_on=["year", "race_category"],
    right_on= ["year", "standardized_category"],
    how="inner"
)

#Combining with Asian and Pacific islander
all_race_combined = pd.concat([broad_merdge, final_asian_island], ignore_index=True)

#calculating the mortality rate for everyone
all_race_combined["mortality_rate_per_100k"] = (all_race_combined["deaths"] / all_race_combined["population"])*100000
all_race_combined["mortality_rate_per_100k"] = all_race_combined["mortality_rate_per_100k"].round(1)

#dropping redundent column
all_race_combined = all_race_combined.drop(columns=["standardized_category", "mortality_per_100k"])

display(all_race_combined.head(7))


,year,race_category,deaths,population,mortality_rate_per_100k
0,2018,American Indian and Alaskan Native,35,4255998.5,0.8
1,2018,Black,588,43939624.5,1.3
2,2018,White,1030,241016624.0,0.4
3,2019,American Indian and Alaskan Native,21,4256268.0,0.5
4,2019,Black,543,44351760.5,1.2
5,2019,White,1057,241354738.5,0.4
6,2020,American Indian and Alaskan Native,34,4234216.0,0.8


In [11]:
#extracting only age groups from clean_age_sex

clean_age = clean_age_sex[clean_age_sex["standardized_category"].isin(["0-24","25-44", "45-64", "65+"])].copy()
clean_age = clean_age.rename(columns={
    "standardized_category": "age_group"
})

clean_age.head()

,year,age_group,population
0,2018,0-24,104008592.0
1,2018,25-44,86843127.0
2,2018,45-64,83892606.0
3,2018,65+,52423114.0
9,2019,0-24,103340955.0


In [12]:
#exporting the only the necessary dataset
export_path_age = "/content/drive/MyDrive/mapping_mortality_gap/data_clean/clean_age_popln.csv"
clean_age.to_csv(export_path_age, index=False)

all_combined_expPath = "/content/drive/MyDrive/mapping_mortality_gap/data_clean/all_race_death_rate_adults_I42.csv"
all_race_combined.to_csv(all_combined_expPath, index=False)
